In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os
from torch.utils.data import DataLoader
import seaborn as sns
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import random
import csv

### Chroma Input

In [2]:
class CNNChromagram(nn.Module):
    def __init__(self, input_time=861):
        super(CNNChromagram, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, 3), padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 3), padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=(3, 3), padding=1)

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=(1, 2))

        reduced_time = input_time // 8
        fc_input_size = 128 * 12

        self.fc1 = nn.Linear(fc_input_size, 64)
        self.fc2 = nn.Linear(64, 2)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)

        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)

        batch_size, channels, freq_bins, time_steps = x.shape
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(batch_size, time_steps, -1)

        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [3]:
class ChromaDataset(torch.utils.data.Dataset):
    def __init__(self, audio_dir, window_size = 20, sr = 22050, hop_length = 512, num_classes = 2):
        self.audio_dir = audio_dir
        self.window_frames = window_size * sr // hop_length
        self.sr = sr
        self.num_classes = num_classes

        self.audio_files = [f for f in os.listdir(audio_dir)]

    def __len__(self):
        return len(self.audio_files)

    def extract_chroma(self, file_path):
        y, _ = librosa.load(file_path)
        chroma = librosa.feature.chroma_stft(y = y, sr = 22050)
        return chroma

    def get_label(self, file_name, num_frames):
        label = np.zeros((num_frames, self.num_classes))

        if 'GMjo' in file_name:
            label[:, 0] = 1
        elif 'Ujo' in file_name:
            label[:, 1] = 1
        return label

    def __getitem__(self, idx):
        file_name = self.audio_files[idx]
        file_path = os.path.join(self.audio_dir, file_name)

        chroma = self.extract_chroma(file_path)
        num_frames = chroma.shape[1]

        labels = self.get_label(file_name, num_frames)

        if num_frames < self.window_frames:
            chroma_pad = np.zeros((12, self.window_frames - num_frames))
            chroma = np.concatenate((chroma, chroma_pad), axis = 1)

            label_pad = np.zeros((self.window_frames - num_frames, self.num_classes))
            labels = np.concatenate((labels, label_pad), axis = 0)

        else:
            start_idx = random.randint(0, num_frames - self.window_frames)
            chroma = chroma[:, start_idx:start_idx + self.window_frames]
            labels = labels[start_idx:start_idx + self.window_frames]

        new_T = chroma.shape[1] // 8
        labels  = torch.tensor(labels, dtype = torch.float32).unsqueeze(0)
        labels = F.interpolate(labels.permute(0, 2, 1), size = new_T, mode = 'linear').permute(0, 2, 1).squeeze(0)

        chroma_tensor = torch.tensor(chroma, dtype = torch.float32).unsqueeze(0)
        label_tensor = torch.tensor(labels, dtype = torch.float32)

        return chroma_tensor, label_tensor

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
audio_dir = "/home/sangheon/Desktop/Pansori_2025_ISMIR/PansoriData/Audio/Vocal"

dataset = ChromaDataset(audio_dir, window_size=20, sr=44100, hop_length=512, num_classes=2)
song_files = dataset.audio_files
loo = LeaveOneOut()

dataset = ChromaDataset(audio_dir, window_size=20, sr=44100, hop_length=512, num_classes=2)
model = CNNChromagram(input_time=dataset.window_frames).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss()

loo_losses = []
loo_accs = []

for train_idx, test_idx in loo.split(song_files):
    train_X, train_y = [], []
    for idx in train_idx:
        chroma, labels = dataset[idx]
        train_X.append(chroma)
        train_y.append(labels)

    val_X, val_y = dataset[test_idx[0]]

    train_X = torch.stack(train_X).to(device)
    train_y = torch.stack(train_y).to(device)

    val_X = val_X.unsqueeze(0).to(device)
    val_y = val_y.unsqueeze(0).to(device)

    train_y = train_y.argmax(dim=-1)
    val_y = val_y.argmax(dim=-1)

    for epoch in range(50):
        optimizer.zero_grad()
        outputs = model(train_X)
        loss = criterion(outputs.permute(0, 2, 1), train_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_output = model(val_X)

        val_loss = criterion(val_output.permute(0, 2, 1), val_y)

        pred = torch.argmax(val_output, dim=2)
        accuracy = (pred == val_y).float().mean().item()

    loo_losses.append(val_loss.item())
    loo_accs.append(accuracy)

    print(f"Validation Loss: {val_loss.item():.4f}, Accuracy: {accuracy:.4f}")

print("\n LOO-CV Results")
print(f"Average Validation Loss: {np.mean(loo_losses):.4f}")
print(f"Average Validation Accuracy: {np.mean(loo_accs):.4f}")

/tmp/ipykernel_438060/3467655752.py:53: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  label_tensor = torch.tensor(labels, dtype = torch.float32)


Validation Loss: 0.0840, Accuracy: 0.9860
Validation Loss: 0.8939, Accuracy: 0.4233
Validation Loss: 0.2420, Accuracy: 0.8884
Validation Loss: 0.5564, Accuracy: 0.7302
Validation Loss: 0.5110, Accuracy: 0.7488
Validation Loss: 0.1790, Accuracy: 0.9302
Validation Loss: 1.0182, Accuracy: 0.6558
Validation Loss: 0.7718, Accuracy: 0.6791
Validation Loss: 0.0884, Accuracy: 0.9674
Validation Loss: 1.0433, Accuracy: 0.6605
Validation Loss: 0.0264, Accuracy: 0.9953
Validation Loss: 0.0109, Accuracy: 0.9953
Validation Loss: 0.0106, Accuracy: 1.0000
Validation Loss: 0.0101, Accuracy: 0.9953
Validation Loss: 0.0131, Accuracy: 1.0000
Validation Loss: 0.0379, Accuracy: 0.9907
Validation Loss: 0.0040, Accuracy: 1.0000
Validation Loss: 0.0127, Accuracy: 0.9953
Validation Loss: 0.0240, Accuracy: 0.9907
Validation Loss: 0.5010, Accuracy: 0.8419
Validation Loss: 0.0069, Accuracy: 0.9953
Validation Loss: 0.9944, Accuracy: 0.7674
Validation Loss: 0.5787, Accuracy: 0.8186
Validation Loss: 0.4803, Accuracy: